In [4]:
import pickle

import pandas as pd
import os

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [5]:
from sklearn.pipeline import make_pipeline

In [6]:
import mlflow

os.environ["AWS_PROFILE"] = 'default'
mlflow.set_tracking_uri("http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/")
mlflow.set_experiment("green-taxi-duration")

2025/06/16 22:21:10 INFO mlflow.tracking.fluent: Experiment with name 'green-taxi-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-artifact-mmichal/2', creation_time=1750112470177, experiment_id='2', last_update_time=1750112470177, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [7]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [11]:
df_train = read_dataframe('../../data/green_tripdata_2023-01.parquet')
df_val = read_dataframe('../../data/green_tripdata_2023-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [13]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    dv = DictVectorizer()
    model = RandomForestRegressor(**params, n_jobs=-1)

    X_train = dv.fit_transform(dict_train)
    model.fit(X_train, y_train)

    X_val = dv.transform(dict_val)
    y_pred = model.predict(X_val)

    rmse = root_mean_squared_error(y_pred, y_val)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(model, artifact_path="model")

    with open('dict_vectorizer.bin', 'wb') as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact('dict_vectorizer.bin')

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 5.39920274232368


2025/06/16 22:30:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run amazing-shrike-260 at: http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/#/experiments/2/runs/f06795cbb3e84dffb4c7447125f67c57
🧪 View experiment at: http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/#/experiments/2


In [15]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/"
RUN_ID = 'f06795cbb3e84dffb4c7447125f67c57'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [16]:
path = client.download_artifacts(run_id=RUN_ID, path='dict_vectorizer.bin')

In [17]:
with open(path, 'rb') as f_in:
    dv = pickle.load(f_in)

In [18]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = root_mean_squared_error(y_pred, y_val)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 5.39920274232368


2025/06/16 23:09:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run enchanting-perch-880 at: http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/#/experiments/2/runs/e0b68d8dd70d4e6fbcaf656cf45a31d5
🧪 View experiment at: http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/#/experiments/2


In [10]:
from mlflow.tracking import MlflowClient


In [20]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = 'b4d3bca8aa8e46a6b8257fe4541b1136'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [21]:
path = client.download_artifacts(run_id=RUN_ID, path='dict_vectorizer.bin')

In [22]:
with open(path, 'rb') as f_out:
    dv = pickle.load(f_out)

In [23]:
dv

DictVectorizer()